# Week 3 — Data contract and leakage check

**Lane:** CTR opportunity scoring  
**Decision moment:** 2026-03-15, after the daily warehouse sync  
**Goal:** rank pages most likely to have CTR below 0.1% during 2026-03-15 through 2026-03-31.

## 1) My lane

I am working in the **CTR opportunity scoring** lane. The useful action is a review queue: pages with the highest predicted probability of very low future CTR should be checked first. This is an opportunity proxy, not a claim that a page is bad or that editing it will cause improvement.

## 2) Contract — five plain-word answers

1. **What one row means:** one pseudonymous client–content page at the 2026-03-15 decision moment, with its search performance summarized from March 1–14.
2. **Table used:** `fact_content_daily_performance`, specifically its `month=2026-03` partition. No join is needed for this first frame.
3. **Time window:** features use 2026-03-01 through 2026-03-14; the outcome uses 2026-03-15 through 2026-03-31. I require at least 100 impressions in each window. June 2026 stays sealed as the final test month.
4. **What I predict or rank:** `low_future_ctr_label = 1` when outcome-window CTR is below 0.1%. I rank eligible pages by predicted probability of that proxy.
5. **What I deliberately exclude:** outcome-window clicks, impressions, and CTR are excluded from honest features because they happen after the decision moment. I also exclude GA4 fields; this slice only needs GSC.

## 3) Three warehouse checks, five features, and the trap

The Hugging Face token is read from the Colab secret named `HF_TOKEN`; it is never printed or stored in this notebook.

In [1]:
from huggingface_hub import hf_hub_download
import duckdb

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except ImportError:
    hf_token = None  # local Hugging Face login, if needed

fact_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token,
)
con = duckdb.connect()
fact = fact_path.replace("'", "''")
print("March 2026 warehouse partition ready.")

March 2026 warehouse partition ready.


### Verification query 1 of 3 — grain

The source grain is one date × client × content row. Aggregating those daily rows by client × content produces the contracted page-level decision rows.

In [2]:
grain_query = f"""
SELECT
    COUNT(*) AS source_daily_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_daily_keys,
    COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_daily_keys,
    COUNT(DISTINCT (client_hash_id, content_hash_id)) AS page_level_keys
FROM read_parquet('{fact}')
WHERE month = '2026-03'
"""
grain_result = con.sql(grain_query).df()
grain_result

,source_daily_rows,distinct_daily_keys,duplicate_daily_keys,page_level_keys
0,9841378,9841378,0,331437


### Verification query 2 of 3 — slice size and date span

In [3]:
span_query = f"""
SELECT
    COUNT(*) AS slice_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS calendar_days
FROM read_parquet('{fact}')
WHERE month = '2026-03'
"""
span_result = con.sql(span_query).df()
span_result

,slice_rows,first_date,last_date,calendar_days
0,9841378,2026-03-01,2026-03-31,31


### Verification query 3 of 3 — GSC availability

Availability is a data contract, not a zero-value test. The filtered CTE deliberately uses `IS TRUE`.

In [4]:
availability_query = f"""
WITH all_rows AS (
    SELECT *
    FROM read_parquet('{fact}')
    WHERE month = '2026-03'
), available_rows AS (
    SELECT *
    FROM all_rows
    WHERE gsc_data_available IS TRUE
)
SELECT
    (SELECT COUNT(*) FROM all_rows) AS rows_before_filter,
    (SELECT COUNT(*) FROM available_rows) AS rows_surviving_is_true,
    ROUND(100.0 * (SELECT COUNT(*) FROM available_rows)
          / (SELECT COUNT(*) FROM all_rows), 2) AS percent_surviving
"""
availability_result = con.sql(availability_query).df()
availability_result

,rows_before_filter,rows_surviving_is_true,percent_surviving
0,9841378,3611061,36.69


### Five-feature frame

All five features stop on 2026-03-14. The later window is used only to make the training label.

1. `early_impressions` — knowable at the decision moment because GSC impressions through March 14 have landed in the daily sync.
2. `early_clicks` — knowable at the decision moment because GSC clicks through March 14 have landed in the daily sync.
3. `early_ctr` — knowable at the decision moment because it uses only early clicks divided by early impressions.
4. `early_avg_position` — knowable at the decision moment because it uses only position totals and impressions observed through March 14.
5. `early_active_days` — knowable at the decision moment because it counts only available daily records through March 14.

In [5]:
feature_query = f"""
WITH early AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)::DOUBLE AS early_impressions,
        SUM(gsc_clicks)::DOUBLE AS early_clicks,
        SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS early_ctr,
        SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS early_avg_position,
        COUNT(DISTINCT report_date)::DOUBLE AS early_active_days
    FROM read_parquet('{fact}')
    WHERE month = '2026-03'
      AND report_date < DATE '2026-03-15'
      AND gsc_data_available IS TRUE
    GROUP BY 1, 2
), outcome AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS future_ctr,
        SUM(gsc_impressions) AS future_impressions
    FROM read_parquet('{fact}')
    WHERE month = '2026-03'
      AND report_date >= DATE '2026-03-15'
      AND gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT
    early.*,
    outcome.future_ctr,
    (outcome.future_ctr < 0.001)::INTEGER AS low_future_ctr_label
FROM early
JOIN outcome USING (client_hash_id, content_hash_id)
WHERE early.early_impressions >= 100
  AND outcome.future_impressions >= 100
"""
feature_frame = con.sql(feature_query).df()
label_source_future_ctr = feature_frame.pop("future_ctr")
print(f"{len(feature_frame):,} eligible page rows")
print(f"Positive-label rate: {feature_frame['low_future_ctr_label'].mean():.3f}")
feature_frame.head()

71,350 eligible page rows
Positive-label rate: 0.459


,client_hash_id,content_hash_id,early_impressions,early_clicks,early_ctr,early_avg_position,early_active_days,low_future_ctr_label
0,client_62f4a7e64f5e0096,content_cfa688665f1b4dad,9642.0,4.0,0.000415,3.689587,14.0,1
1,client_62f4a7e64f5e0096,content_4fdbf66afc9394d1,1048.0,1.0,0.000954,2.975191,14.0,0
2,client_62f4a7e64f5e0096,content_56c8b7686fc9f816,155.0,0.0,0.000000,14.967742,14.0,1
3,client_62f4a7e64f5e0096,content_9b867d8fc00d438f,150.0,1.0,0.006667,2.960000,14.0,1
4,client_9958f0a7ae1df715,content_3046965022e9f691,114.0,0.0,0.000000,49.605263,14.0,1


### Honest quick score

Balanced accuracy gives equal weight to both label classes. This random split is only a quick leakage demonstration; the modeling weeks should use time-based validation.

In [6]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

honest_features = [
    "early_impressions",
    "early_clicks",
    "early_ctr",
    "early_avg_position",
    "early_active_days",
]
model_frame = feature_frame[
    ["client_hash_id", "content_hash_id", *honest_features, "low_future_ctr_label"]
].copy()
train_idx, test_idx = train_test_split(
    model_frame.index,
    test_size=0.25,
    random_state=42,
    stratify=model_frame["low_future_ctr_label"],
)

def quick_score(columns):
    model = DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=100, random_state=42
    )
    model.fit(model_frame.loc[train_idx, columns], model_frame.loc[train_idx, "low_future_ctr_label"])
    predictions = model.predict(model_frame.loc[test_idx, columns])
    return balanced_accuracy_score(model_frame.loc[test_idx, "low_future_ctr_label"], predictions)

honest_score = quick_score(honest_features)
print(f"Honest balanced accuracy: {honest_score:.3f}")

Honest balanced accuracy: 0.728


### The trap — add the answer in disguise, then delete it

`label_derived_future_ctr` is the exact outcome used to define the label. It is unavailable on March 15, so adding it should make the score look unrealistically perfect.

In [7]:
model_frame["label_derived_future_ctr"] = label_source_future_ctr  # deliberate leakage
leaked_score = quick_score([*honest_features, "label_derived_future_ctr"])
print(f"Honest balanced accuracy: {honest_score:.3f}")
print(f"Leaked balanced accuracy: {leaked_score:.3f}")

del model_frame["label_derived_future_ctr"]
assert "label_derived_future_ctr" not in model_frame
print("Leak deleted. The number I keep is the honest score above.")

Honest balanced accuracy: 0.728
Leaked balanced accuracy: 1.000
Leak deleted. The number I keep is the honest score above.


## 4) Named limitation

**Limitation — no query or seasonality context:** this page-level slice cannot tell whether low CTR comes from the page itself, its search-query mix, SERP features, branded demand, or normal seasonality. The score can prioritize review; it cannot establish why CTR is low or prove that an edit will improve it.

## 5) Self-check

In [8]:
assert int(grain_result.loc[0, "duplicate_daily_keys"]) == 0
assert span_result.loc[0, "first_date"].date().isoformat() == "2026-03-01"
assert span_result.loc[0, "last_date"].date().isoformat() == "2026-03-31"
assert "gsc_data_available IS TRUE" in availability_query
assert int(availability_result.loc[0, "rows_surviving_is_true"]) > 0
assert len(honest_features) == 5
assert "future_ctr" not in feature_frame.columns
assert "label_derived_future_ctr" not in model_frame.columns
assert leaked_score > honest_score

print("✓ Five contract answers")
print("✓ Exactly three verification queries, including IS TRUE availability")
print("✓ Five decision-time features")
print("✓ Deliberate leak demonstrated and removed")
print("✓ One named limitation")

✓ Five contract answers
✓ Exactly three verification queries, including IS TRUE availability
✓ Five decision-time features
✓ Deliberate leak demonstrated and removed
✓ One named limitation
